# Unit 2 Assignment: Mixture of Experts (MoE) Router
### Smart Customer Support Routing System


In [5]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

print("Groq Client Loaded Successfully ✅")

Groq Client Loaded Successfully ✅


In [6]:
MODEL_CONFIG = {
    "technical": {
        "model": "llama-3.1-8b-instant",
        "system_prompt": """You are a Technical Support Expert.
Be rigorous, precise, and code-focused.
Provide debugging explanations and corrected code when needed."""
    },
    "billing": {
        "model": "llama-3.1-8b-instant",
        "system_prompt": """You are a Billing Support Expert.
Be empathetic, financial-focused, and policy-driven.
Help users clearly with charges, refunds, and subscriptions."""
    },
    "general": {
        "model": "llama-3.1-8b-instant",
        "system_prompt": """You are a General Support Assistant.
Answer casual and general questions in a friendly way."""
    }
}

In [7]:
def route_prompt(user_input):
    routing_instruction = f"""
Classify this text into one of these categories:
[technical, billing, general]

Return ONLY the category name.

Text: {user_input}
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0,   # VERY IMPORTANT
        messages=[
            {"role": "user", "content": routing_instruction}
        ]
    )

    return response.choices[0].message.content.strip().lower().replace(".", "")

In [8]:
# Step 4 - Orchestrator Function

def process_request(user_input):
    category = route_prompt(user_input)

    if category not in MODEL_CONFIG:
        category = "general"

    expert_config = MODEL_CONFIG[category]

    response = client.chat.completions.create(
        model=expert_config["model"],
        temperature=0.7,   # VERY IMPORTANT
        messages=[
            {"role": "system", "content": expert_config["system_prompt"]},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content


In [11]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

print("Client ready ✅")

Client ready ✅


In [9]:
print("ROUTER TEST - BILLING\n")

user_query = "I was charged twice for my subscription this month."

category = route_prompt(user_query)

print("User Query:", user_query)
print("Predicted Category:", category)

print("\nFinal Response:\n")
print(process_request(user_query))

ROUTER TEST - BILLING

User Query: I was charged twice for my subscription this month.
Predicted Category: billing

Final Response:

I'm so sorry to hear that you were charged twice for your subscription. That can be frustrating and confusing. I'm here to help you figure out what went wrong and get it sorted out.

To better understand the situation, can you please tell me:

1. What is the date of the duplicate charge?
2. What is the amount of the duplicate charge?
3. What is your subscription plan and payment method?
4. When did you first notice the duplicate charge?

This information will help me investigate the issue and find a solution for you.

Also, I'll need to check on a few things behind the scenes. This might take a few minutes, but I'll do my best to get everything sorted out for you as quickly as possible.

In the meantime, I want to reassure you that we take situations like this seriously and will do our best to prevent it from happening again in the future.

Please let me 

In [10]:
print("ROUTER TEST - TECHNICAL\n")

user_query = "My python script is throwing an IndexError on line 5."

category = route_prompt(user_query)

print("User Query:", user_query)
print("Predicted Category:", category)

print("\nFinal Response:\n")
print(process_request(user_query))

ROUTER TEST - TECHNICAL

User Query: My python script is throwing an IndexError on line 5.
Predicted Category: technical

Final Response:

I'd be happy to help you debug your Python script. To do so, I'll need more information. Please provide the following:

1. The full stacktrace (error message) of the IndexError, including the line number and the error message.
2. The code snippet that's causing the error (line 5).
3. Any relevant code surrounding line 5.

However, if you'd like to provide a minimal reproducible example, I can assist you in debugging the issue.

Here's a general example of how to debug an IndexError in Python:

```python
# Example list
my_list = [1, 2, 3]

# Attempting to access an index out of range
try:
    print(my_list[3])  # This will raise an IndexError
except IndexError as e:
    print(f"Error: {e}")
```

The error message will indicate that you're trying to access an index that's out of range. For example:

```
Error: list index out of range
```

In this case